# CineEmbed — EDA v2

Pipeline-first refresh of the EDA notebook. Applies all 13 fixes from the audit (see `docs/superpowers/specs/2026-05-03-eda-v2-design.md`). Produces a clean (329044, 451) feature matrix.

**Sections:**
- §1 Setup & Reproducibility
- §2 Pipeline Function Definitions
- §3 Pipeline Execution
- §4 EDA Visualizations
- §5 Persistence


In [ ]:
# §1 — Setup & Reproducibility
import os, json, hashlib, random, warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, MultiLabelBinarizer
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold
from scipy import stats

# Optional GPU stack — only imported if available
try:
    import torch
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    if HAS_TORCH:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


CONFIG = {
    'seed': 42,
    'data_dir': Path('data'),
    'artifacts_dir': Path('artifacts'),
    'figures_dir': Path('artifacts/figures'),

    # File paths
    'paths': {
        'details': Path('data/AllMoviesDetailsCleaned.csv'),
        'casting': Path('data/AllMoviesCastingRaw.csv'),
        'awards':  Path('data/220k_awards_by_directors.csv'),
    },

    # Feature engineering knobs (single source of truth — fix #11 + clean ablation)
    'top_n_genres': 20,
    'top_n_languages': 30,
    'q99_clip_threshold': 0.99,
    'runtime_clip': (10, 300),

    # Embedding model (fix #5 — multilingual)
    'embedding_model': 'paraphrase-multilingual-MiniLM-L12-v2',
    'embedding_batch_size': 64,
    'embedding_dim': 384,
    'embedding_cache': Path('artifacts/text_embeddings.npy'),
    'embedding_meta': Path('artifacts/text_embeddings.meta.json'),
}

CONFIG['artifacts_dir'].mkdir(parents=True, exist_ok=True)
CONFIG['figures_dir'].mkdir(parents=True, exist_ok=True)

seed_everything(CONFIG['seed'])

# Reproducibility self-check
_check = np.random.rand(3)
print("\u2705 \u00a71 Setup complete")
print(f"   seed = {CONFIG['seed']}")
print(f"   np.random sample (deterministic) = {_check}")
print(f"   torch available = {HAS_TORCH}")


## §2.1 — Data Layer
Pure functions: `load_csvs`, `normalize_director_name`, `merge_details_casting`. Each function has a sanity-test cell immediately after.

In [ ]:
# §2.1 — Data layer functions
import unicodedata

def normalize_director_name(name: str | None) -> str:
    """Stable director key for joins.

    Steps:
      1. None / NaN / empty → ''
      2. Unicode NFKD decompose, strip combining marks (accents)
      3. Swap "Last, First" → "First Last"
      4. Lowercase, collapse internal whitespace, strip ends.
    """
    if name is None or (isinstance(name, float) and np.isnan(name)):
        return ''
    s = str(name).strip()
    if not s:
        return ''
    # NFKD + ascii filter (drops accents)
    s = unicodedata.normalize('NFKD', s)
    s = ''.join(ch for ch in s if not unicodedata.combining(ch))
    # "Last, First" → "First Last"
    if ',' in s:
        parts = [p.strip() for p in s.split(',', 1)]
        if len(parts) == 2 and parts[0] and parts[1]:
            s = f"{parts[1]} {parts[0]}"
    # collapse whitespace, lowercase
    s = ' '.join(s.split()).lower()
    return s


In [ ]:
# Test §2.1: normalize_director_name (fix #8)
_cases = [
    ('Steven Spielberg',   'steven spielberg'),
    ('Spielberg, Steven',  'steven spielberg'),
    ('  Pedro  Almodóvar ', 'pedro almodovar'),
    ('Léa  Pool',          'lea pool'),
    (None,                 ''),
    ('',                   ''),
    ('SCORSESE, MARTIN',   'martin scorsese'),
]
for raw, expected in _cases:
    got = normalize_director_name(raw)
    assert got == expected, f"normalize_director_name({raw!r}) = {got!r} != {expected!r}"
print(f"\u2705 normalize_director_name: {len(_cases)} cases pass")


In [ ]:
def load_csvs(paths: dict[str, Path]) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Load the three source CSVs with their correct separators.

    details, casting use ';' (TMDB-derived); awards uses ','.
    """
    details = pd.read_csv(paths['details'], sep=';', low_memory=False)
    casting = pd.read_csv(paths['casting'], sep=';', low_memory=False)
    awards  = pd.read_csv(paths['awards'],  sep=',', low_memory=False)
    return details, casting, awards


In [ ]:
def merge_details_casting(details: pd.DataFrame, casting: pd.DataFrame) -> pd.DataFrame:
    """Inner-shape merge on `id`. Adds `director_name_norm` (fix #8) for award joins.

    Carries `director_name` (raw) for human readability and `director_name_norm`
    for joins. Only the necessary casting columns are pulled in to avoid bloat.
    """
    casting_slim = casting[['id', 'director_name', 'director_gender']].copy()
    casting_slim['director_name_norm'] = casting_slim['director_name'].apply(normalize_director_name)
    merged = details.merge(casting_slim, on='id', how='left')
    return merged


In [ ]:
# Test §2.1: merge_details_casting
_details = pd.DataFrame({'id': [1, 2, 3], 'title': ['A', 'B', 'C']})
_casting = pd.DataFrame({
    'id': [1, 2, 3],
    'director_name': ['Steven Spielberg', 'Spielberg, Steven', None],
    'director_gender': [2, 2, 0],
})
_merged = merge_details_casting(_details, _casting)
assert set(_merged.columns) >= {'id', 'title', 'director_name', 'director_name_norm', 'director_gender'}
assert _merged.loc[0, 'director_name_norm'] == 'steven spielberg'
assert _merged.loc[1, 'director_name_norm'] == 'steven spielberg'   # normalized form merges
assert _merged.loc[2, 'director_name_norm'] == ''
print("✅ merge_details_casting passes")


## §2.2 — Awards Layer
Temporal-aware per-film aggregation. Implements fixes #1 (temporal leak) and #9 (Oscar/Palme regex word-boundary).

In [ ]:
import re

# OSCAR_RE: Word-boundary match for "Oscar" with a negative lookahead that
# excludes "Oscar Wilde Award" (Oscar Wilde is a person, not the Oscar award).
# The plan's audit identified this as a real-world false-positive trap.
OSCAR_RE = re.compile(r'\bOscar\b(?!\s+Wilde)', flags=re.IGNORECASE)
PALME_RE = re.compile(r'\bPalme\b', flags=re.IGNORECASE)


def aggregate_awards_temporal(
    awards: pd.DataFrame,
    films: pd.DataFrame,
) -> pd.DataFrame:
    """Per-film aggregation of director awards.

    Two fixes:
      - #1 (temporal): only awards with year <= film.release_year contribute.
      - #9 (regex):    Oscar/Palme detection uses \\b word-boundary.

    Returns a DataFrame with one row per film id and columns:
      prior_total_nominations, prior_total_wins,
      prior_oscar_nominations, prior_oscar_wins,
      prior_palme_nominations, prior_palme_wins

    Note: *_nominations counts entries where the director was nominated but did
    NOT win (i.e., is_oscar/is_palme AND NOT is_won). *_wins counts entries
    where they won (is_oscar/is_palme AND is_won).
    """
    # Normalize director key on awards side
    awards = awards.copy()
    awards['director_name_norm'] = awards['director_name'].apply(normalize_director_name)

    # Year column (if missing, parse from a date-like field; here we trust 'year')
    awards['year'] = pd.to_numeric(awards['year'], errors='coerce')
    awards = awards.dropna(subset=['year'])
    awards['year'] = awards['year'].astype(int)

    # Annotate flags up-front (avoid per-row regex during the join)
    awards['is_won']   = (awards['outcome'].fillna('') == 'Won')
    awards['is_oscar'] = awards['category'].fillna('').str.contains(OSCAR_RE, regex=True)
    awards['is_palme'] = awards['category'].fillna('').str.contains(PALME_RE, regex=True)

    # Films side — derive year
    films = films.copy()
    films['release_year'] = pd.to_datetime(films['release_date'], errors='coerce').dt.year

    # For each (director_norm), pre-sort awards ascending by year for cumulative aggregation
    award_groups = awards.sort_values('year').groupby('director_name_norm')

    rows = []
    for _, film in films[['id', 'director_name_norm', 'release_year']].iterrows():
        out = {
            'id': film['id'],
            'prior_total_nominations': 0,
            'prior_total_wins': 0,
            'prior_oscar_nominations': 0,
            'prior_oscar_wins': 0,
            'prior_palme_nominations': 0,
            'prior_palme_wins': 0,
        }
        director = film['director_name_norm']
        year = film['release_year']
        if not director or pd.isna(year):
            rows.append(out)
            continue
        if director not in award_groups.groups:
            rows.append(out)
            continue
        sub = award_groups.get_group(director)
        sub = sub[sub['year'] <= year]
        if sub.empty:
            rows.append(out)
            continue
        out['prior_total_nominations']  = int((~sub['is_won']).sum())
        out['prior_total_wins']         = int(sub['is_won'].sum())
        out['prior_oscar_nominations']  = int((sub['is_oscar'] & ~sub['is_won']).sum())
        out['prior_oscar_wins']         = int((sub['is_oscar'] &  sub['is_won']).sum())
        out['prior_palme_nominations']  = int((sub['is_palme'] & ~sub['is_won']).sum())
        out['prior_palme_wins']         = int((sub['is_palme'] &  sub['is_won']).sum())
        rows.append(out)

    return pd.DataFrame(rows)


In [ ]:
# Test §2.2: aggregate_awards_temporal
# Synthetic awards: director "alice" has 2 wins (1995, 2010), 1 nom (2005)
# Director "bob" has 1 oscar win (2000) and 1 "Oscar Wilde Award" nom (1998 — false positive trap)
_awards = pd.DataFrame({
    'director_name': ['Alice', 'Alice', 'Alice', 'Bob', 'Bob'],
    'category':      ['Best Director', 'Best Picture', 'Best Director — Oscar',
                      'Academy Award (Oscar)', 'Oscar Wilde Award'],
    'outcome':       ['Won', 'Nominated', 'Won', 'Won', 'Nominated'],
    'year':          [1995, 2005, 2010, 2000, 1998],
})
_films = pd.DataFrame({
    'id': [101, 102, 103, 104],
    'director_name_norm': ['alice', 'alice', 'bob', 'bob'],
    'release_date': pd.to_datetime(['1990-01-01', '2008-01-01', '1995-01-01', '2005-01-01']),
})
_agg = aggregate_awards_temporal(_awards, _films)

# Assertions (one per fix):
# Fix #1 — temporal: alice 1990 film sees 0 wins; alice 2008 film sees 1 win (1995)
row_a1990 = _agg.set_index('id').loc[101]
row_a2008 = _agg.set_index('id').loc[102]
assert row_a1990['prior_total_wins'] == 0,  f"1990 leak: {row_a1990['prior_total_wins']}"
assert row_a2008['prior_total_wins'] == 1,  f"2008 expects 1 win, got {row_a2008['prior_total_wins']}"

# Fix #9 — word-boundary: bob 1995 sees 0 oscar wins (Wilde shouldn't count, win is 2000)
# bob 2005 sees 1 oscar win (2000) and 0 oscar noms (Wilde is filtered out)
row_b1995 = _agg.set_index('id').loc[103]
row_b2005 = _agg.set_index('id').loc[104]
assert row_b1995['prior_oscar_wins'] == 0
assert row_b2005['prior_oscar_wins'] == 1
assert row_b2005['prior_oscar_nominations'] == 0, \
    f"Wilde should not count, got {row_b2005['prior_oscar_nominations']}"

print("\u2705 aggregate_awards_temporal: temporal cutoff + regex word-boundary work")
